# Notebook 6 — Statistical Reporting and Paper Table Packaging
### Detectability-Aware Post-Merger GW Anomaly Screening

**This notebook performs no new experiments.** It loads the already-saved, final `test_block` results
from `nb5_artifacts/test_block_records.json` (and `FINAL_SUMMARY.json`) and adds:

1. Wilson (binomial) 95% confidence intervals for conditional/end-to-end TPR/FPR (ridge, lsq, power),
   per-tau Stage-1 pass rates, and per-tau Stage-2 conditional flag rates.
2. A compact final paper table: `Method | Mode | TPR | FPR | AUC | Notes`.
3. A per-tau breakdown table: `tau | class | Stage-1 pass rate | LSQ flag rate | ridge flag rate | power flag rate`.

No injection is drawn, no model is fit, no threshold is touched. `test_block` is not re-loaded --
everything here is a deterministic function of the already-saved per-sample records.

**Preserved final interpretation (do not deviate from this in the tables or text below):**
- Stage 1 detects structured local time-frequency excess, not physical anomalies.
- LSQ is the strongest Stage-2 discriminator.
- Ridge is a secondary, TF-native, interpretable method.
- Power is a weaker but real, direction-sensitive baseline.
- No dark matter / dark-sector detection claim.
- Signal model is a toy single-mode damped sinusoid.
- L1 only, no multi-detector claim.
- `test_block` was used once, after the design freeze.


In [1]:
# ---------------------------------------------------------------------------
# 0. Load already-saved final results -- no new experiments
# ---------------------------------------------------------------------------
import os
import json
import numpy as np
from google.colab import drive; drive.mount('/content/drive')

NB5_DIR = '/content/drive/MyDrive/gw_postmerger/nb5_artifacts'
OUT_DIR = '/content/drive/MyDrive/gw_postmerger/nb6_artifacts'
os.makedirs(OUT_DIR, exist_ok=True)

with open(os.path.join(NB5_DIR, 'test_block_records.json')) as f:
    test_records = json.load(f)

with open(os.path.join(NB5_DIR, 'FINAL_SUMMARY.json')) as f:
    final_summary = json.load(f)

print(f"Loaded {len(test_records)} test_block records (no re-sampling, no re-fitting).")

TAU_GRID_MS = final_summary['frozen_config_used']['evaluation_grid']['tau_grid_ms']
STANDARD_TAUS_MS = set(final_summary['frozen_config_used']['evaluation_grid']['standard_taus_ms'])
ANOMALOUS_TAUS_MS = set(final_summary['frozen_config_used']['evaluation_grid']['anomalous_taus_ms'])
METHOD_LABELS = {'ridge': 'Ridge/energy-decay', 'lsq': 'Nonlinear LSQ', 'power': 'Power (calibrated logistic)'}


Mounted at /content/drive
Loaded 1800 test_block records (no re-sampling, no re-fitting).


## 1. Wilson (binomial) 95% confidence interval

Used throughout below in place of a normal-approximation interval, since several of the rates being
reported (e.g. LSQ's 0.000 false-flag rate at several tau values) sit at or near 0/1, where the normal
approximation is invalid.


In [2]:
# ---------------------------------------------------------------------------
# 1. Wilson score interval
# ---------------------------------------------------------------------------
def wilson_ci(k, n, z=1.959963984540054):  # z for 95%
    if n == 0:
        return (float('nan'), float('nan'))
    phat = k / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))

def rate_with_ci(k, n):
    if n == 0:
        return float('nan'), (float('nan'), float('nan'))
    p = k / n
    lo, hi = wilson_ci(k, n)
    return p, (lo, hi)

def fmt_rate(p, ci):
    if np.isnan(p):
        return "--"
    return f"{p:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]"

## 2. Recompute rates + Wilson CIs from the saved records (conditional and end-to-end)


In [3]:
# ---------------------------------------------------------------------------
# 2. Rates + CIs, conditional and end-to-end, per method
# ---------------------------------------------------------------------------
# Re-derive the same flag decisions already made in Notebook 5, purely by reading the fields already
# saved on each record (tau_hat_ridge, tau_hat_lsq, total_power_raw were only ever computed once, in
# Notebook 5 -- no re-computation of any score happens here). Frozen thresholds, restated from
# FINAL_SUMMARY's embedded frozen_config for self-containedness.
FROZEN = final_summary['frozen_config_used']
RIDGE_THRESH_S = FROZEN['stage2_secondary']['anomaly_threshold_ms'] / 1000.0
LSQ_THRESH_S = FROZEN['stage2_primary']['anomaly_threshold_ms'] / 1000.0

# The power flag decision requires the fitted classifier's probability, which was not persisted
# per-sample in test_block_records.json (only total_power_raw was saved). We reconstruct the flag
# from the ALREADY-COMPUTED operating-point TPR/FPR in FINAL_SUMMARY instead of re-deriving a
# probability -- i.e. we use FINAL_SUMMARY's own recorded flag counts directly wherever available,
# falling back to tau_hat-threshold logic only for ridge/lsq (which need no classifier).
def flag_ridge(r):
    return r.get('tau_hat_ridge') is not None and r['tau_hat_ridge'] < RIDGE_THRESH_S

def flag_lsq(r):
    return r.get('tau_hat_lsq') is not None and r['tau_hat_lsq'] < LSQ_THRESH_S

def compute_counts(records_subset, flag_fn, denom_mode):
    anom = [r for r in records_subset if r['is_anomalous']]
    std = [r for r in records_subset if not r['is_anomalous']]
    if denom_mode == 'conditional':
        anom = [r for r in anom if r['stage1_pass']]
        std = [r for r in std if r['stage1_pass']]
    n_anom, n_std = len(anom), len(std)
    k_tp = sum(1 for r in anom if r['stage1_pass'] and flag_fn(r))
    k_fp = sum(1 for r in std if r['stage1_pass'] and flag_fn(r))
    return k_tp, n_anom, k_fp, n_std

results_table1 = []
for method_name, flag_fn in [('ridge', flag_ridge), ('lsq', flag_lsq)]:
    for mode in ['conditional', 'end_to_end']:
        k_tp, n_anom, k_fp, n_std = compute_counts(test_records, flag_fn, mode)
        tpr, tpr_ci = rate_with_ci(k_tp, n_anom)
        fpr, fpr_ci = rate_with_ci(k_fp, n_std)
        results_table1.append({'method': method_name, 'mode': mode, 'tpr': tpr, 'tpr_ci': tpr_ci,
                                'fpr': fpr, 'fpr_ci': fpr_ci, 'n_anom': n_anom, 'n_std': n_std,
                                'k_tp': k_tp, 'k_fp': k_fp})
        print(f"{method_name:6s} {mode:12s} TPR={fmt_rate(tpr, tpr_ci)} ({k_tp}/{n_anom})   "
              f"FPR={fmt_rate(fpr, fpr_ci)} ({k_fp}/{n_std})")

# Power: use the operating-point TPR/FPR + sample sizes already recorded in FINAL_SUMMARY (Notebook 5
# Section 7), recomputing only the Wilson CI here (also new statistical packaging, not a new experiment).
for row in final_summary['conditional_and_endtoend_metrics']:
    if row['method'] != 'power':
        continue
    n_anom, n_std = row['n_anom'], row['n_std']
    k_tp = round(row['tpr'] * n_anom) if not np.isnan(row['tpr']) else 0
    k_fp = round(row['fpr'] * n_std) if not np.isnan(row['fpr']) else 0
    tpr, tpr_ci = rate_with_ci(k_tp, n_anom)
    fpr, fpr_ci = rate_with_ci(k_fp, n_std)
    results_table1.append({'method': 'power', 'mode': row['mode'], 'tpr': tpr, 'tpr_ci': tpr_ci,
                            'fpr': fpr, 'fpr_ci': fpr_ci, 'n_anom': n_anom, 'n_std': n_std,
                            'k_tp': k_tp, 'k_fp': k_fp})
    print(f"power  {row['mode']:12s} TPR={fmt_rate(tpr, tpr_ci)} ({k_tp}/{n_anom})   "
          f"FPR={fmt_rate(fpr, fpr_ci)} ({k_fp}/{n_std})")

with open(os.path.join(OUT_DIR, 'rates_with_ci.json'), 'w') as f:
    json.dump(results_table1, f, indent=1, default=str)
print(f"\nSaved -> {OUT_DIR}/rates_with_ci.json")


ridge  conditional  TPR=0.486 [0.446, 0.526] (289/595)   FPR=0.016 [0.010, 0.025] (16/1028)
ridge  end_to_end   TPR=0.482 [0.442, 0.522] (289/600)   FPR=0.013 [0.008, 0.022] (16/1200)
lsq    conditional  TPR=0.879 [0.850, 0.903] (523/595)   FPR=0.012 [0.007, 0.020] (12/1028)
lsq    end_to_end   TPR=0.872 [0.843, 0.896] (523/600)   FPR=0.010 [0.006, 0.017] (12/1200)
power  conditional  TPR=0.054 [0.038, 0.075] (32/595)   FPR=0.010 [0.005, 0.018] (10/1028)
power  end_to_end   TPR=0.053 [0.038, 0.074] (32/600)   FPR=0.008 [0.005, 0.015] (10/1200)

Saved -> /content/drive/MyDrive/gw_postmerger/nb6_artifacts/rates_with_ci.json


## 3. Per-tau Stage-1 pass rates and Stage-2 conditional flag rates, with Wilson CIs


In [4]:
# ---------------------------------------------------------------------------
# 3. Per-tau breakdown with CIs
# ---------------------------------------------------------------------------
per_tau_results = []
for tau_ms in TAU_GRID_MS:
    subset = [r for r in test_records if r['tau_ms'] == tau_ms]
    n_total_tau = len(subset)
    k_pass = sum(r['stage1_pass'] for r in subset)
    stage1_rate, stage1_ci = rate_with_ci(k_pass, n_total_tau)

    row = {'tau_ms': tau_ms, 'class': 'anomalous' if tau_ms in ANOMALOUS_TAUS_MS else 'standard',
           'stage1_pass_rate': stage1_rate, 'stage1_pass_ci': stage1_ci,
           'stage1_n': n_total_tau, 'stage1_k': k_pass}

    for method_name, flag_fn in [('ridge', flag_ridge), ('lsq', flag_lsq)]:
        k_tp, n_anom, k_fp, n_std = compute_counts(subset, flag_fn, 'conditional')
        if tau_ms in ANOMALOUS_TAUS_MS:
            rate, ci, k, n = *rate_with_ci(k_tp, n_anom), k_tp, n_anom
        else:
            rate, ci, k, n = *rate_with_ci(k_fp, n_std), k_fp, n_std
        row[f'{method_name}_flag_rate'] = rate
        row[f'{method_name}_flag_ci'] = ci
        row[f'{method_name}_flag_k'] = k
        row[f'{method_name}_flag_n'] = n

    per_tau_results.append(row)
    print(f"tau={tau_ms:3d}ms ({row['class']:9s}): "
          f"Stage1={fmt_rate(stage1_rate, stage1_ci)} ({k_pass}/{n_total_tau})  "
          f"ridge={fmt_rate(row['ridge_flag_rate'], row['ridge_flag_ci'])}  "
          f"lsq={fmt_rate(row['lsq_flag_rate'], row['lsq_flag_ci'])}")

# Power per-tau: reconstruct counts from FINAL_SUMMARY's per_tau_flag_rates (already computed in
# Notebook 5; only the CI is newly added here).
power_per_tau = final_summary['per_tau_flag_rates'].get('power', {})
for row in per_tau_results:
    tau_key = str(row['tau_ms'])
    p_rate = power_per_tau.get(tau_key)
    if p_rate is not None:
        n = row['stage1_k'] if row['class'] == 'anomalous' else (row['stage1_n'] - 0)  # n_std-equivalent
        # n for the power conditional rate at this tau is the same Stage-1-passing count used above
        n_for_power = row['ridge_flag_n']
        k_for_power = round(p_rate * n_for_power)
        p_final, p_ci = rate_with_ci(k_for_power, n_for_power)
        row['power_flag_rate'] = p_final
        row['power_flag_ci'] = p_ci
        row['power_flag_k'] = k_for_power
        row['power_flag_n'] = n_for_power

with open(os.path.join(OUT_DIR, 'per_tau_with_ci.json'), 'w') as f:
    json.dump(per_tau_results, f, indent=1, default=str)
print(f"\nSaved -> {OUT_DIR}/per_tau_with_ci.json")


tau=  1ms (anomalous): Stage1=0.987 [0.966, 0.995] (296/300)  ridge=0.736 [0.684, 0.783]  lsq=0.932 [0.898, 0.956]
tau=  3ms (anomalous): Stage1=0.997 [0.981, 0.999] (299/300)  ridge=0.237 [0.193, 0.289]  lsq=0.826 [0.779, 0.865]
tau=  5ms (standard ): Stage1=0.973 [0.948, 0.986] (292/300)  ridge=0.024 [0.012, 0.049]  lsq=0.041 [0.024, 0.070]
tau= 10ms (standard ): Stage1=0.937 [0.903, 0.959] (281/300)  ridge=0.014 [0.006, 0.036]  lsq=0.000 [0.000, 0.013]
tau= 20ms (standard ): Stage1=0.853 [0.809, 0.889] (256/300)  ridge=0.008 [0.002, 0.028]  lsq=0.000 [0.000, 0.015]
tau= 40ms (standard ): Stage1=0.663 [0.608, 0.714] (199/300)  ridge=0.015 [0.005, 0.043]  lsq=0.000 [0.000, 0.019]

Saved -> /content/drive/MyDrive/gw_postmerger/nb6_artifacts/per_tau_with_ci.json


## 4. Table 1 — compact final paper table

`Method | Mode | TPR | FPR | AUC | Notes`. AUC is only meaningful for the conditional
(Stage-1-passing) set, so it is populated on conditional rows only. AUC values are copied from
`FINAL_SUMMARY.json` (already computed in Notebook 5 -- not recomputed here).


In [5]:
# ---------------------------------------------------------------------------
# 4. Table 1
# ---------------------------------------------------------------------------
AUC_LOOKUP = {
    'ridge': final_summary['auc_results'].get('ridge (tau_hat, inverted)'),
    'lsq': final_summary['auc_results'].get('lsq (tau_hat, inverted)'),
    'power': final_summary['auc_results'].get('power (AUC(logistic))'),
}
NOTES = {
    'ridge': 'TF-native, interpretable, secondary Stage-2 method',
    'lsq': 'Strongest Stage-2 discriminator (primary)',
    'power': 'Weaker, direction-sensitive baseline; AUC(power)=%.3f, AUC(-power)=%.3f' % (
        final_summary['auc_results'].get('power (AUC(power))', float('nan')),
        final_summary['auc_results'].get('power (AUC(-power))', float('nan'))),
}

print(f"{'Method':22s} {'Mode':12s} {'TPR':>20s} {'FPR':>20s} {'AUC':>7s}  Notes")
table1_rows = []
for row in results_table1:
    auc_str = f"{AUC_LOOKUP[row['method']]:.3f}" if row['mode'] == 'conditional' and AUC_LOOKUP[row['method']] is not None else "--"
    tpr_str = fmt_rate(row['tpr'], row['tpr_ci'])
    fpr_str = fmt_rate(row['fpr'], row['fpr_ci'])
    notes = NOTES[row['method']] if row['mode'] == 'conditional' else ''
    print(f"{METHOD_LABELS[row['method']]:22s} {row['mode']:12s} {tpr_str:>20s} {fpr_str:>20s} {auc_str:>7s}  {notes}")
    table1_rows.append({'method': METHOD_LABELS[row['method']], 'mode': row['mode'],
                         'tpr': tpr_str, 'fpr': fpr_str, 'auc': auc_str, 'notes': notes})

import csv
with open(os.path.join(OUT_DIR, 'table1_final.csv'), 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['method', 'mode', 'tpr', 'fpr', 'auc', 'notes'])
    writer.writeheader()
    writer.writerows(table1_rows)
print(f"\nSaved -> {OUT_DIR}/table1_final.csv")


Method                 Mode                          TPR                  FPR     AUC  Notes
Ridge/energy-decay     conditional  0.486 [0.446, 0.526] 0.016 [0.010, 0.025]   0.930  TF-native, interpretable, secondary Stage-2 method
Ridge/energy-decay     end_to_end   0.482 [0.442, 0.522] 0.013 [0.008, 0.022]      --  
Nonlinear LSQ          conditional  0.879 [0.850, 0.903] 0.012 [0.007, 0.020]   0.943  Strongest Stage-2 discriminator (primary)
Nonlinear LSQ          end_to_end   0.872 [0.843, 0.896] 0.010 [0.006, 0.017]      --  
Power (calibrated logistic) conditional  0.054 [0.038, 0.075] 0.010 [0.005, 0.018]   0.726  Weaker, direction-sensitive baseline; AUC(power)=0.274, AUC(-power)=0.726
Power (calibrated logistic) end_to_end   0.053 [0.038, 0.074] 0.008 [0.005, 0.015]      --  

Saved -> /content/drive/MyDrive/gw_postmerger/nb6_artifacts/table1_final.csv


## 5. Table 2 — per-tau breakdown


In [6]:
# ---------------------------------------------------------------------------
# 5. Table 2
# ---------------------------------------------------------------------------
print(f"{'tau (ms)':9s} {'class':10s} {'Stage-1 pass rate':>20s} {'LSQ flag rate':>20s} "
      f"{'Ridge flag rate':>20s} {'Power flag rate':>20s}")
table2_rows = []
for row in per_tau_results:
    s1_str = fmt_rate(row['stage1_pass_rate'], row['stage1_pass_ci'])
    lsq_str = fmt_rate(row['lsq_flag_rate'], row['lsq_flag_ci'])
    ridge_str = fmt_rate(row['ridge_flag_rate'], row['ridge_flag_ci'])
    power_str = fmt_rate(row.get('power_flag_rate', float('nan')), row.get('power_flag_ci', (float('nan'), float('nan'))))
    print(f"{row['tau_ms']:<9d} {row['class']:10s} {s1_str:>20s} {lsq_str:>20s} {ridge_str:>20s} {power_str:>20s}")
    table2_rows.append({'tau_ms': row['tau_ms'], 'class': row['class'], 'stage1_pass_rate': s1_str,
                         'lsq_flag_rate': lsq_str, 'ridge_flag_rate': ridge_str, 'power_flag_rate': power_str})

with open(os.path.join(OUT_DIR, 'table2_per_tau.csv'), 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['tau_ms', 'class', 'stage1_pass_rate', 'lsq_flag_rate',
                                            'ridge_flag_rate', 'power_flag_rate'])
    writer.writeheader()
    writer.writerows(table2_rows)
print(f"\nSaved -> {OUT_DIR}/table2_per_tau.csv")


tau (ms)  class         Stage-1 pass rate        LSQ flag rate      Ridge flag rate      Power flag rate
1         anomalous  0.987 [0.966, 0.995] 0.932 [0.898, 0.956] 0.736 [0.684, 0.783] 0.054 [0.034, 0.086]
3         anomalous  0.997 [0.981, 0.999] 0.826 [0.779, 0.865] 0.237 [0.193, 0.289] 0.054 [0.033, 0.085]
5         standard   0.973 [0.948, 0.986] 0.041 [0.024, 0.070] 0.024 [0.012, 0.049] 0.021 [0.009, 0.044]
10        standard   0.937 [0.903, 0.959] 0.000 [0.000, 0.013] 0.014 [0.006, 0.036] 0.011 [0.004, 0.031]
20        standard   0.853 [0.809, 0.889] 0.000 [0.000, 0.015] 0.008 [0.002, 0.028] 0.004 [0.001, 0.022]
40        standard   0.663 [0.608, 0.714] 0.000 [0.000, 0.019] 0.015 [0.005, 0.043] 0.000 [0.000, 0.019]

Saved -> /content/drive/MyDrive/gw_postmerger/nb6_artifacts/table2_per_tau.csv


## 6. LaTeX table source (booktabs style, matching the manuscript's table conventions)

Ready to paste directly into the manuscript.


In [7]:
# ---------------------------------------------------------------------------
# 6. LaTeX table generation
# ---------------------------------------------------------------------------
def tex_escape(s):
    return str(s).replace('_', '\\_').replace('%', '\\%')

latex_table1 = []
latex_table1.append(r"\begin{table*}[htbp]")
latex_table1.append(r"    \centering")
latex_table1.append(r"    \caption{Final \texttt{test\_block} evaluation: conditional and end-to-end "
                     r"performance for the frozen Stage-2 methods and the power baseline. TPR/FPR are "
                     r"reported with Wilson 95\% confidence intervals. AUC is defined only on the "
                     r"Stage-1-passing (conditional) set.}")
latex_table1.append(r"    \label{tab:final_results}")
latex_table1.append(r"    \begin{tabular}{@{}llllll@{}}")
latex_table1.append(r"        \toprule")
latex_table1.append(r"        \textbf{Method} & \textbf{Mode} & \textbf{TPR [95\% CI]} & "
                     r"\textbf{FPR [95\% CI]} & \textbf{AUC} & \textbf{Notes} \\")
latex_table1.append(r"        \midrule")
for row in table1_rows:
    mode_disp = 'Conditional' if row['mode'] == 'conditional' else 'End-to-end'
    latex_table1.append(f"        {tex_escape(row['method'])} & {mode_disp} & "
                         f"{tex_escape(row['tpr'])} & {tex_escape(row['fpr'])} & "
                         f"{tex_escape(row['auc'])} & {tex_escape(row['notes'])} \\\\")
latex_table1.append(r"        \bottomrule")
latex_table1.append(r"    \end{tabular}")
latex_table1.append(r"\end{table*}")
latex_table1_str = "\n".join(latex_table1)
print(latex_table1_str)

latex_table2 = []
latex_table2.append(r"\begin{table*}[htbp]")
latex_table2.append(r"    \centering")
latex_table2.append(r"    \caption{Per-tau breakdown on \texttt{test\_block}: Stage-1 pass rate and "
                     r"Stage-2 conditional flag rate for each method, with Wilson 95\% confidence "
                     r"intervals. For anomalous $\tau$, the flag rate is a recall (higher is better); "
                     r"for standard $\tau$, it is a false-flag rate (lower is better).}")
latex_table2.append(r"    \label{tab:per_tau}")
latex_table2.append(r"    \begin{tabular}{@{}llllll@{}}")
latex_table2.append(r"        \toprule")
latex_table2.append(r"        $\boldsymbol{\tau}$ \textbf{(ms)} & \textbf{Class} & "
                     r"\textbf{Stage-1 pass rate} & \textbf{LSQ flag rate} & "
                     r"\textbf{Ridge flag rate} & \textbf{Power flag rate} \\")
latex_table2.append(r"        \midrule")
for row in table2_rows:
    latex_table2.append(f"        {row['tau_ms']} & {tex_escape(row['class'])} & "
                         f"{tex_escape(row['stage1_pass_rate'])} & {tex_escape(row['lsq_flag_rate'])} & "
                         f"{tex_escape(row['ridge_flag_rate'])} & {tex_escape(row['power_flag_rate'])} \\\\")
latex_table2.append(r"        \bottomrule")
latex_table2.append(r"    \end{tabular}")
latex_table2.append(r"\end{table*}")
latex_table2_str = "\n".join(latex_table2)
print()
print(latex_table2_str)

with open(os.path.join(OUT_DIR, 'table1_final.tex'), 'w') as f:
    f.write(latex_table1_str)
with open(os.path.join(OUT_DIR, 'table2_per_tau.tex'), 'w') as f:
    f.write(latex_table2_str)
print(f"\n\nSaved -> {OUT_DIR}/table1_final.tex, table2_per_tau.tex")


\begin{table*}[htbp]
    \centering
    \caption{Final \texttt{test\_block} evaluation: conditional and end-to-end performance for the frozen Stage-2 methods and the power baseline. TPR/FPR are reported with Wilson 95\% confidence intervals. AUC is defined only on the Stage-1-passing (conditional) set.}
    \label{tab:final_results}
    \begin{tabular}{@{}llllll@{}}
        \toprule
        \textbf{Method} & \textbf{Mode} & \textbf{TPR [95\% CI]} & \textbf{FPR [95\% CI]} & \textbf{AUC} & \textbf{Notes} \\
        \midrule
        Ridge/energy-decay & Conditional & 0.486 [0.446, 0.526] & 0.016 [0.010, 0.025] & 0.930 & TF-native, interpretable, secondary Stage-2 method \\
        Ridge/energy-decay & End-to-end & 0.482 [0.442, 0.522] & 0.013 [0.008, 0.022] & -- &  \\
        Nonlinear LSQ & Conditional & 0.879 [0.850, 0.903] & 0.012 [0.007, 0.020] & 0.943 & Strongest Stage-2 discriminator (primary) \\
        Nonlinear LSQ & End-to-end & 0.872 [0.843, 0.896] & 0.010 [0.006, 0.017] & -- &

## 7. Preserved final interpretation (restated for the record)

- **Stage 1** detects structured local time-frequency excess, not physical anomalies -- it is not shown
  to measure post-merger coherence, ridge structure, or any specific physical mechanism.
- **LSQ is the strongest Stage-2 discriminator** -- highest AUC and, more importantly, the largest
  low-FPR operating-point recall, especially on the harder tau=3ms case.
- **Ridge is a secondary, TF-native, interpretable method** -- competitive ranking-level separation,
  weaker operating-point recall than LSQ, no further tuning performed.
- **Power is a weaker but real, direction-sensitive baseline** -- moderate AUC, poor operating-point
  recall; must never be reported as "uninformative."
- **No dark matter / dark-sector detection claim** anywhere in this project's results.
- **Signal model is a toy single-mode damped sinusoid** -- not a physically complete post-merger
  waveform.
- **L1 only** -- no multi-detector coincidence claim.
- **`test_block` was used once, after the design freeze** -- these are this project's final,
  non-negotiable numbers; no further tuning follows this notebook.

```
nb6_artifacts/
  rates_with_ci.json, per_tau_with_ci.json   # full numeric results with CIs
  table1_final.csv, table2_per_tau.csv       # spreadsheet-ready tables
  table1_final.tex, table2_per_tau.tex       # paste-ready LaTeX tables (booktabs style)
```

**Next: manuscript drafting**, using these tables directly.
